<a href="https://colab.research.google.com/github/Syed-Bilal-Hussaini/AI-Chatbot-for-Mental-Health-Support/blob/main/AI_Chatbot_for_Mental_Health_Support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch flask

In [ ]:
def generate_response(user_input, chat_history_ids=None):
    # Encode the new user input with EOS token
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt').to(device)

    # Append new input to existing chat history
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    # Keep only the last 512 tokens to prevent hitting maximum sequence limit (1024)
    if bot_input_ids.shape[-1] > 512:
        bot_input_ids = bot_input_ids[:, -512:]

    # Generate model outputs
    output_ids = model.generate(
        bot_input_ids,
        max_length=bot_input_ids.shape[-1] + 50,  # Allow 50 new tokens past the input
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    # Decode ONLY the newly generated tokens (slice past input length)
    input_length = bot_input_ids.shape[-1]
    new_tokens = output_ids[0][input_length:]

    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Fallback if model generates empty output
    if not response:
        response = "I'm listening. Tell me more about what's on your mind."

    return response, output_ids

In [ ]:
!pip install flask-cors pyngrok

In [ ]:
#This code block doesn't work completely, look into it again.
def safety_filter(user_input):
    # Basic keyword trigger system
    crisis_keywords = ["suicide", "harm", "kill myself", "end it all"]

    if any(word in user_input.lower() for word in crisis_keywords):
        return ("I'm really sorry you're feeling this way, but I'm an AI and not equipped to help in a crisis. "
                "Please reach out to a professional or a crisis hotline immediately (e.g., 988 in the US).")
    return None

import random
def apply_empathy_prefix(response):
    # Simple logic to soften the AI output
    prefixes = ["I hear you. ", "I'm here for you. ", "That sounds tough. "]
    # You could use random.choice(prefixes) here
    return f"{random.choice(prefixes)}{response}"

In [ ]:
import threading
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Load Model & Tokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium").to(device)

# 2. Generation Logic
def generate_response(user_input, chat_history_ids=None):
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt').to(device)
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    # Keep tensor within context window limit
    if bot_input_ids.shape[-1] > 512:
        bot_input_ids = bot_input_ids[:, -512:]

    output_ids = model.generate(
        bot_input_ids,
        max_length=bot_input_ids.shape[-1] + 50,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    input_length = bot_input_ids.shape[-1]
    new_tokens = output_ids[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return response, output_ids

# 3. Flask App Setup
app = Flask(__name__)
CORS(app)
session_history = None

@app.route("/chat", methods=["POST"])
def chat():
    global session_history
    data = request.get_json(force=True)
    user_msg = data.get("message", "") if data else ""

    safety_msg = safety_filter(user_msg)
    if safety_msg:
        return jsonify({"response": safety_msg})
    # print(f"\n[Incoming User Message]: {user_msg}")

    # Generate output via DialoGPT
    response_text, session_history = generate_response(user_msg, session_history)

    # print(f"[Model Output]: {response_text}")
    final_response = apply_empathy_prefix(response_text)

    return jsonify({"response": final_response})

# 4. Expose with Ngrok & Run Thread
ngrok.kill()
ngrok.set_auth_token("30YB6uJnSxTFTgPOfLh6Nrv9zHj_41RrKqwdXg7H8eTjd2ax8") # Replace with your ngrok authtoken
tunnel = ngrok.connect(5000)

print(f"\n * Endpoint Active: {tunnel.public_url}/chat\n")

def start_flask():
    app.run(port=5000, use_reloader=False)

threading.Thread(target=start_flask).start()

Using device: cpu


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  863MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


 * Endpoint Active: https://ce49-34-50-169-69.ngrok-free.app/chat



In [ ]:
# !pip install flask-cors pyngrok

# import threading
# from flask import Flask, request, jsonify
# from flask_cors import CORS
# from pyngrok import ngrok
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch

# # Load pre-trained model and tokenizer
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model_name = "microsoft/DialoGPT-medium"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name)

# # Kill any existing tunnels
# ngrok.kill()

# # Set your Ngrok Auth Token
# NGROK_AUTH_TOKEN = "30YB6uJnSxTFTgPOfLh6Nrv9zHj_41RrKqwdXg7H8eTjd2ax8"  # Paste your new token here
# ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# app = Flask(__name__)
# CORS(app)

# # Global variable to store session history
# session_history = None

# # Placeholder function for AI response generation
# def generate_response(user_msg, history):
#     # Add your model/LLM logic here
#     reply = f"AI Support: I understand you said '{user_msg}'. How can I help further?"
#     history = (history or []) + [{"user": user_msg, "bot": reply}]
#     return reply, history

# @app.route("/")
# def home():
#     return "Flask Server is Running! Send POST requests to /chat to interact."

# @app.route("/chat", methods=["POST"])
# def chat():
#     global session_history
#     data = request.json or {}
#     user_msg = data.get("message", "")

#     # Get AI response
#     response_text, session_history = generate_response(user_msg, session_history)

#     return jsonify({"response": response_text})

# # 1. Start the ngrok tunnel on port 5000
# tunnel = ngrok.connect(5000)
# print(f" * Public URL: {tunnel.public_url}")
# print(f" * Chat Endpoint: {tunnel.public_url}/chat")

# # 2. Run Flask in a background thread so it doesn't block Colab
# def start_flask():
#     app.run(port=5000, use_reloader=False)

# threading.Thread(target=start_flask).start()

In [ ]:
%%html
<!DOCTYPE html>
<html>
<head>
    <title>Mental Health AI</title>
    <style>
        body { font-family: sans-serif; display: flex; flex-direction: column; align-items: center; background: #f0f2f5; }
        #chatbox { width: 400px; height: 500px; border: 1px solid #ccc; background: white; overflow-y: scroll; padding: 20px; border-radius: 10px; }
        .user { color: blue; text-align: right; }
        .bot { color: green; text-align: left; }
    </style>
</head>
<body>
    <h2>AI Counselor</h2>
    <div id="chatbox"></div>
    <div style="margin-top: 10px;">
        <input type="text" id="userInput" placeholder="Type here...">
        <button onclick="send()">Send</button>
    </div>

    <script>
        async function send() {
            const text = document.getElementById('userInput').value;
            const box = document.getElementById('chatbox');
            box.innerHTML += `<p class="user"><b>You:</b> ${text}</p>`;

            // PASTE YOUR NGORK PUBLIC URL HERE
            const res = await fetch('https://d086-34-80-28-109.ngrok-free.app/chat', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify({message: text})
            });
            const data = await res.json();
            box.innerHTML += `<p class="bot"><b>Bot:</b> ${data.response}</p>`;
            document.getElementById('userInput').value = '';
        }
    </script>
</body>
</html>